# Lendo o arquivo bruto


In [0]:
# Lendo arquivo csv que está no datalake
vendas = spark.read.format("csv").option("header", "true").option("header", "true").option("delimiter", ",").option("encoding", "utf-8").load("/Volumes/lh_nautical/datalake/datas/vendas_2023_2024.csv")

vendasLines = vendas.count()
#Verificando se o arquivo csv foi lido corretamente
vendas.show()

# Acessando o Catalog lh_nautical e o Schema bronze_lh_nautical


In [0]:
%sql
-- Informando o catalogo e schema
USE CATALOG lh_nautical;USE SCHEMA bronze_lh_nautical;

# Salvando o arquivo bruto como tabela delta na camada Bronze

In [0]:
#Salvando como tabela no schema bronze
vendas.write.mode("overwrite").format("delta").saveAsTable("lh_nautical.bronze_lh_nautical.tbl_vendas")

# Conferindo a tabela delta criada na camada Bronze

In [0]:
#Conferindo se a tabela foi criada corretamente
tbl_vendas = spark.read.table("bronze_lh_nautical.tbl_vendas")
tbl_vendas.display()


# Recriando a tabela para camada Silver

### - Vendo se tem nulo


In [0]:
#Verificando se há dados nulos
tbl_vendas.filter(tbl_vendas.id.isNull()).display()
tbl_vendas.filter(tbl_vendas.id_client.isNull()).display()
tbl_vendas.filter(tbl_vendas.id_product.isNull()).display()
tbl_vendas.filter(tbl_vendas.qtd.isNull()).display()
tbl_vendas.filter(tbl_vendas.total.isNull()).display()
tbl_vendas.filter(tbl_vendas.sale_date.isNull()).display()


### - Modificando os tipos das colunas

In [0]:
# Importando bibliotecas
from pyspark.sql.types import *
from pyspark.sql.functions import when, col
#Verificando o tipo dos dados
tbl_vendas_clean.printSchema()

#Modificando os tipos dos dados da tabela e salvando como tabela no schema silver
tbl_vendas_clean.withColumn("id", col("id").cast(IntegerType())).withColumn("id_client", col("id_client").cast(IntegerType())).withColumn("id_product", col("id_product").cast(IntegerType())).withColumn("qtd", col("qtd").cast(IntegerType())).withColumn("total", col("total").cast(DecimalType(10,2))).write.mode("overwrite").format("delta").saveAsTable("silver_lh_nautical.tbl_vendas")
#Conferindo se a tabela foi criada corretamente
tbl_vendas_clean = spark.read.table("silver_lh_nautical.tbl_vendas")
tbl_vendas_clean.display()

### - Tratando a coluna sale_date

In [0]:
%sql
-- Conferindo se o tamanho de todas as datas é igual a 10
select distinct len(sale_date) from lh_nautical.silver_lh_nautical.tbl_vendas

### - Descobrindo qual dado começa com o ano e qual começa com o dia

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim, lower, regexp_replace, length, substring_index

tbl_vendas_clean = spark.read.table("silver_lh_nautical.tbl_vendas")
# Modificando os dados da coluna sale_date para o formato yyyy-MM-dd e modificando o tipo da coluna para date
tbl_vendas_clean = (tbl_vendas_clean
                    .withColumn("sale_date", 
                                F.when(F.length(F.substring_index(tbl_vendas_clean.sale_date, '-', 1)) == 2, F.date_format(F.to_date(tbl_vendas_clean.sale_date, 'dd-MM-yyyy'), 'yyyy-MM-dd'))
                                .otherwise(F.date_format(tbl_vendas_clean.sale_date, 'yyyy-MM-dd')).cast("date")
                                )
                    )

#Conferindo a modificação
tbl_vendas_clean.display()
tbl_vendas_clean.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_lh_nautical.tbl_vendas")